# 04 - Ablation Studies

This notebook evaluates the robustness and reproducibility of the DenseNet-Attention model
through systematic ablation studies:

1. Data augmentation level (none / basic / standard / heavy)
2. Training set size (25% / 50% / 75% / 100%)
3. Model components (attention module, focal loss)
4. Multiple runs with different random seeds

In [ ]:
import sys
sys.path.insert(0, "..")

import copy
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import get_dataloaders
from src.models import get_model
from src.train import run_experiment
from src.evaluate import evaluate_model, compute_metrics
from src.utils import set_seed, get_device, load_config, save_results

plt.rcParams["figure.dpi"] = 100
sns.set_style("whitegrid")

config = load_config("../configs/default.yaml")
device = get_device()
DATA_ROOT = "../data/chest_xray"
SEED = config["reproducibility"]["seed"]
SEEDS = config["reproducibility"]["seeds_for_multiple_runs"]

print(f"Device: {device}")
print(f"Seeds for multi-run experiments: {SEEDS}")

In [ ]:
def run_single_ablation(model_kwargs, data_kwargs, run_config, experiment_name, seed=42):
    """Train a single model and return test metrics."""
    set_seed(seed)

    dataloaders = get_dataloaders(
        DATA_ROOT,
        num_workers=0,
        seed=seed,
        **data_kwargs,
    )

    model = get_model("densenet_attention", **model_kwargs)

    history = run_experiment(
        model_name="densenet_attention",
        model=model,
        dataloaders=dataloaders,
        device=device,
        config=run_config,
        experiment_name=experiment_name,
    )

    test_results = evaluate_model(model, dataloaders["test"], device)
    return test_results["metrics"], history

## Ablation 1: Data Augmentation Level

We train the DenseNet-Attention model with four different augmentation strategies
to quantify the impact of data augmentation on generalization.

In [ ]:
aug_levels = ["none", "basic", "standard", "heavy"]
aug_results = {}

for level in aug_levels:
    print(f"\n{'='*60}")
    print(f"Training with augmentation: {level}")
    print(f"{'='*60}")

    data_kwargs = {
        "augmentation": level,
        "image_size": config["data"]["image_size"],
        "batch_size": config["training"]["batch_size"],
        "val_split": config["data"]["val_split"],
    }
    model_kwargs = {
        "pretrained": True,
        "dropout": config["model"]["dropout"],
        "use_attention": True,
    }

    metrics, _ = run_single_ablation(
        model_kwargs, data_kwargs, config,
        experiment_name=f"ablation_aug_{level}",
        seed=SEED,
    )
    aug_results[level] = metrics
    print(f"  AUROC: {metrics['auroc']:.4f}, F1: {metrics['f1_macro']:.4f}")

In [ ]:
aug_df = pd.DataFrame([
    {"Augmentation": level, "AUROC": r["auroc"], "F1 (macro)": r["f1_macro"],
     "Sensitivity": r["sensitivity"], "Specificity": r["specificity"]}
    for level, r in aug_results.items()
])
print(aug_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(aug_df["Augmentation"], aug_df["AUROC"], color="#4C72B0")
axes[0].set_ylabel("AUROC")
axes[0].set_title("AUROC by Augmentation Level")
axes[0].set_ylim(0.8, 1.0)

axes[1].bar(aug_df["Augmentation"], aug_df["F1 (macro)"], color="#DD8452")
axes[1].set_ylabel("F1 (macro)")
axes[1].set_title("F1 Score by Augmentation Level")
axes[1].set_ylim(0.7, 1.0)

plt.tight_layout()
plt.savefig("../results/ablation_augmentation.png", bbox_inches="tight")
plt.show()

## Ablation 2: Training Set Size

We evaluate how performance degrades as the amount of training data decreases.
This is relevant in medical imaging where labeled data is often scarce.

In [ ]:
fractions = [0.25, 0.50, 0.75, 1.0]
size_results = {}

for frac in fractions:
    print(f"\n{'='*60}")
    print(f"Training with {frac*100:.0f}% of training data")
    print(f"{'='*60}")

    data_kwargs = {
        "augmentation": "standard",
        "image_size": config["data"]["image_size"],
        "batch_size": config["training"]["batch_size"],
        "val_split": config["data"]["val_split"],
        "train_fraction": frac,
    }
    model_kwargs = {
        "pretrained": True,
        "dropout": config["model"]["dropout"],
        "use_attention": True,
    }

    metrics, _ = run_single_ablation(
        model_kwargs, data_kwargs, config,
        experiment_name=f"ablation_size_{int(frac*100)}",
        seed=SEED,
    )
    size_results[frac] = metrics
    print(f"  AUROC: {metrics['auroc']:.4f}, F1: {metrics['f1_macro']:.4f}")

In [ ]:
size_df = pd.DataFrame([
    {"Train Fraction": f"{frac*100:.0f}%", "AUROC": r["auroc"],
     "F1 (macro)": r["f1_macro"], "Sensitivity": r["sensitivity"],
     "Specificity": r["specificity"]}
    for frac, r in size_results.items()
])
print(size_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
x = [f"{f*100:.0f}%" for f in fractions]
ax.plot(x, [size_results[f]["auroc"] for f in fractions], "o-", label="AUROC", linewidth=2)
ax.plot(x, [size_results[f]["f1_macro"] for f in fractions], "s-", label="F1 (macro)", linewidth=2)
ax.plot(x, [size_results[f]["sensitivity"] for f in fractions], "^-", label="Sensitivity", linewidth=2)
ax.set_xlabel("Training Data Fraction")
ax.set_ylabel("Score")
ax.set_title("Performance vs. Training Set Size")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.7, 1.0)

plt.tight_layout()
plt.savefig("../results/ablation_data_size.png", bbox_inches="tight")
plt.show()

## Ablation 3: Model Components

We study the contribution of two key design choices:
- Channel attention module
- Focal loss (vs. weighted BCE)

In [ ]:
component_configs = {
    "Full model (attention + focal)": {"use_attention": True, "use_focal": True},
    "No attention": {"use_attention": False, "use_focal": True},
    "No focal loss": {"use_attention": True, "use_focal": False},
    "No attention, no focal": {"use_attention": False, "use_focal": False},
}

component_results = {}

for name, comp_cfg in component_configs.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    data_kwargs = {
        "augmentation": "standard",
        "image_size": config["data"]["image_size"],
        "batch_size": config["training"]["batch_size"],
        "val_split": config["data"]["val_split"],
    }
    model_kwargs = {
        "pretrained": True,
        "dropout": config["model"]["dropout"],
        "use_attention": comp_cfg["use_attention"],
    }

    run_config = copy.deepcopy(config)
    run_config["model"]["use_focal_loss"] = comp_cfg["use_focal"]

    safe_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace(",", "")
    metrics, _ = run_single_ablation(
        model_kwargs, data_kwargs, run_config,
        experiment_name=f"ablation_component_{safe_name}",
        seed=SEED,
    )
    component_results[name] = metrics
    print(f"  AUROC: {metrics['auroc']:.4f}, F1: {metrics['f1_macro']:.4f}")

In [ ]:
comp_df = pd.DataFrame([
    {"Configuration": name, "AUROC": r["auroc"], "F1 (macro)": r["f1_macro"],
     "Sensitivity": r["sensitivity"], "Specificity": r["specificity"]}
    for name, r in component_results.items()
])
print(comp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(component_results))
width = 0.25

names = list(component_results.keys())
aurocs = [component_results[n]["auroc"] for n in names]
f1s = [component_results[n]["f1_macro"] for n in names]
sens = [component_results[n]["sensitivity"] for n in names]

ax.bar(x - width, aurocs, width, label="AUROC")
ax.bar(x, f1s, width, label="F1 (macro)")
ax.bar(x + width, sens, width, label="Sensitivity")

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Score")
ax.set_title("Model Component Ablation")
ax.legend()
ax.set_ylim(0.7, 1.0)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("../results/ablation_components.png", bbox_inches="tight")
plt.show()

## Multiple Runs with Different Seeds

To assess variability, we run the full DenseNet-Attention model 5 times
with different random initializations.

In [ ]:
multi_run_results = []

for i, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"Run {i+1}/{len(SEEDS)} with seed={seed}")
    print(f"{'='*60}")

    data_kwargs = {
        "augmentation": "standard",
        "image_size": config["data"]["image_size"],
        "batch_size": config["training"]["batch_size"],
        "val_split": config["data"]["val_split"],
    }
    model_kwargs = {
        "pretrained": True,
        "dropout": config["model"]["dropout"],
        "use_attention": True,
    }

    metrics, _ = run_single_ablation(
        model_kwargs, data_kwargs, config,
        experiment_name=f"multirun_seed_{seed}",
        seed=seed,
    )
    multi_run_results.append({"seed": seed, **metrics})
    print(f"  AUROC: {metrics['auroc']:.4f}, F1: {metrics['f1_macro']:.4f}")

In [ ]:
mr_df = pd.DataFrame(multi_run_results)

print("Per-run results:")
print(mr_df[["seed", "auroc", "f1_macro", "sensitivity", "specificity"]].to_string(index=False))

print("\nAggregated statistics:")
for col in ["auroc", "f1_macro", "sensitivity", "specificity", "npv"]:
    mean = mr_df[col].mean()
    std = mr_df[col].std()
    print(f"  {col:>15s}: {mean:.4f} +/- {std:.4f}")

In [ ]:
metrics_to_plot = ["auroc", "f1_macro", "sensitivity", "specificity"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(14, 4))

for i, metric in enumerate(metrics_to_plot):
    values = mr_df[metric].values
    axes[i].boxplot(values, widths=0.4)
    axes[i].scatter(np.ones(len(values)), values, alpha=0.6, s=40, zorder=3)
    axes[i].set_title(metric.upper())
    axes[i].set_ylabel("Score")
    axes[i].set_xticks([])
    axes[i].grid(True, alpha=0.3, axis="y")

plt.suptitle("Variability Across 5 Random Seeds", fontsize=13)
plt.tight_layout()
plt.savefig("../results/ablation_multirun.png", bbox_inches="tight")
plt.show()

## Save all ablation results

In [ ]:
all_ablation_results = {
    "augmentation_ablation": {k: v for k, v in aug_results.items()},
    "data_size_ablation": {str(k): v for k, v in size_results.items()},
    "component_ablation": component_results,
    "multi_run": multi_run_results,
    "multi_run_summary": {
        col: {"mean": float(mr_df[col].mean()), "std": float(mr_df[col].std())}
        for col in ["auroc", "f1_macro", "sensitivity", "specificity", "npv"]
    },
}

save_results(all_ablation_results, "ablation_results", output_dir="../results")
print("All ablation results saved.")